In [1]:
from transformers import AutoModelForSequenceClassification
import torch
dataset_name = "imdb"
tokenizer_checkpoint = "DeepWokLab/bert-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"


model = AutoModelForSequenceClassification.from_pretrained("DeepWokLab/bert-tiny")

print(model)
print(model.device)

/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepWokLab/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-1

In [2]:
import torch
import time
from chop.models import get_model
from chop.dataset import get_dataset_info


def timed_gpu(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, start.elapsed_time(end) / 1000


def timed_cpu(fn):
    start = time.time()
    result = fn()
    return result, time.time() - start


def get_data():
    query = torch.ones(32, 8, 128, 64, dtype=torch.float16)
    key = torch.ones(32, 8, 128, 64, dtype=torch.float16)
    value = torch.ones(32, 8, 128, 64, dtype=torch.float16)
    return query, key, value



def time_model(fn, n=1000, device="cpu"):
    times = []
    query, key, value = get_data()
    query = query.to(device)
    key = key.to(device)
    value = value.to(device)
    for _ in range(n):
        if device == "cpu":
            _, t = timed_cpu(lambda: fn(query.cpu(), key.cpu(), value.cpu()))
        else:
            _, t = timed_gpu(lambda: fn(query, key, value))
        times.append(t)
    avg_time = sum(times) / len(times)
    return avg_time

/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


## SDPA Profiling

We can profile the fused softmax kernel to estimate the overhead

In [3]:
import math
import functools
import torch
import torch.nn.functional as F
from torch import Tensor
from torch.nn.attention.flex_attention import flex_attention, create_block_mask


class ScaledDotProductAttention(torch.nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, query, key, value):
        scale_factor = 1 / math.sqrt(query.size(-1))
        score = query @ key.transpose(-2, -1) * scale_factor
        attn = F.softmax(score, -1)
        context = attn @ value
        return context


class ScaledDotProductAttentionFused(torch.nn.Module):
    def forward(self, query, key, value):
        return F.scaled_dot_product_attention(query, key, value)


class ScaledDotProductAttentionFlex(torch.nn.Module):
    def forward(self, query, key, value):
        seq_len = query.size(-2)
        block_mask = _get_sliding_window_block_mask(seq_len, 64, str(query.device))
        orig_dtype = query.dtype
        if orig_dtype == torch.float16:
            query, key, value = query.float(), key.float(), value.float()
        return flex_attention(query, key, value, block_mask=block_mask).to(orig_dtype)



@functools.lru_cache(maxsize=None)
def _get_sliding_window_block_mask(seq_len, window_size, device):
    def mask_fn(b, h, q_idx, kv_idx):
        return (q_idx - kv_idx).abs() <= window_size

    return create_block_mask(
        mask_fn, B=None, H=None,
        Q_LEN=seq_len, KV_LEN=seq_len,
        device=device,
    )


def sliding_window_flex_attention(query, key, value, attn_mask, dropout_p, is_causal):
    seq_len = query.size(-2)
    block_mask = _get_sliding_window_block_mask(seq_len, 64, str(query.device))
    # flex_attention requires float32 or bfloat16
    orig_dtype = query.dtype
    if orig_dtype == torch.float16:
        query, key, value = query.float(), key.float(), value.float()
    return flex_attention(query, key, value, block_mask=block_mask).to(orig_dtype)




class PropogateValue(torch.nn.Module):
    def forward(self, query: Tensor, key: Tensor, value: Tensor):
        output = torch.ones(query.size(0), query.size(1), query.size(2), value.size(-1), dtype=torch.float16, device=query.device)
        return output


In [4]:
import functools

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
sdpa_naive = ScaledDotProductAttention()
sdpa_fused = ScaledDotProductAttentionFused()


print("Naive time:        ", time_model(sdpa_naive, device=device))
print("Fused time:        ", time_model(sdpa_fused, device=device))


Using device: cuda
Naive time:         0.0015918786640167222
Fused time:         0.00020170019158721123


### Profiling in model

We can run a transform pass to remove the SDPA operators in the model, and profile the overhead from softmax in Bert-Tiny.

In [5]:
import os
import platform

# Add Homebrew path for macOS (Graphviz is typically in system PATH on Linux)
if platform.system() == 'Darwin':  # macOS
    homebrew_bin = '/opt/homebrew/bin'  # ARM64 Mac
    if not os.path.exists(homebrew_bin):
        homebrew_bin = '/usr/local/bin'  # Intel Mac
    if os.path.exists(homebrew_bin):
        os.environ['PATH'] = homebrew_bin + ':' + os.environ.get('PATH', '')

from chop import MaseGraph
model.config.problem_type = "single_label_classification"

mg = MaseGraph(
    model,
    hf_input_names=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

mg.draw("bert-base-uncased.svg")

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.


In [6]:
import torch
import chop.passes as passes
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

dummy_input = tokenizer(
    [
        "AI may take over the world one day",
        "This is why you should learn ADLS",
    ],
    return_tensors="pt",
)

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg)

INFO     Getting dummy input for DeepWokLab/bert-tiny.


INFO     Getting dummy input for DeepWokLab/bert-tiny.


tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]],


        [[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]]])
tensor([[[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       

In [7]:
import torch.fx as fx
from chop.tools import get_logger

logger = get_logger("mase_logger")

# def remove_scaled_dot_product_attention_transform_pass(mg, pass_args={}):

#     for node in mg.fx_graph.nodes:
#         if node.op == "call_function" and "scaled_dot_product_attention" in node.name:
#             logger.info(f"Removing scaled_dot_product_attention module: {node.target}")

#             # Create and register a replacement module that propagates shape
#             new_mod_name = f"propogate_value_{node.name}"
#             i = 0
#             while hasattr(mg.model, new_mod_name):
#                 i += 1
#                 new_mod_name = f"propogate_value_{node.name}_{i}"

#             mg.model.add_function(new_mod_name, propogate_value)

#             # Insert a new call_module node using the same args as the SDPA node
#             with mg.fx_graph.inserting_before(node):
#                 replacement_node = mg.fx_graph.call_function(new_mod_name, args=node.args, kwargs=node.kwargs)

#             # Replace all users of the scaled_dot_product_attention node with the new module node
#             node.replace_all_uses_with(replacement_node)

#             # Erase the scaled_dot_product_attention node
#             mg.fx_graph.erase_node(node)
#         else:
#             logger.debug(f"Skipping node: {node.target}")

#     return mg, {}

def remove_scaled_dot_product_attention_transform_pass(mg, pass_args={}):
    for node in mg.fx_graph.nodes:
        if node.op == "call_function" and "scaled_dot_product_attention" in node.name:
            logger.info(f"Removing scaled_dot_product_attention module: {node.target}")
            with mg.fx_graph.inserting_before(node):
                replacement_node = mg.fx_graph.call_function(
                    sliding_window_flex_attention, args=node.args, kwargs=node.kwargs
                )
            node.replace_all_uses_with(replacement_node)
            mg.fx_graph.erase_node(node)
        else:
            logger.debug(f"Skipping node: {node.target}")
    return mg, {}


In [8]:
import functools
import torch.fx as fx
from chop.tools import get_logger

logger = get_logger("mase_logger")

def replace_sdpa_with_sliding_window_flex_attention_transform_pass(mg, pass_args={}):
    """Replace scaled_dot_product_attention nodes with sliding window flex attention.

    pass_args:
        window_size (int): tokens each position can attend to on either side (default: 64).
    """
    window_size = pass_args.get("window_size", 64)
    replacement_fn = functools.partial(sliding_window_flex_attention, window_size=window_size)

    for node in list(mg.fx_graph.nodes):
        if node.op == "call_function" and "scaled_dot_product_attention" in node.name:
            logger.info(
                f"Replacing {node.target} with sliding window flex attention (window={window_size})"
            )
            with mg.fx_graph.inserting_before(node):
                replacement_node = mg.fx_graph.call_function(
                    replacement_fn, args=node.args, kwargs=node.kwargs
                )
            node.replace_all_uses_with(replacement_node)
            mg.fx_graph.erase_node(node)
        else:
            logger.debug(f"Skipping node: {node.target}")

    return mg, {}


### Performer (FAVOR+) Attention

[Performers](https://arxiv.org/abs/2009.14794) (Choromanski et al., 2021) approximate softmax attention in **O(n·m·d)** time (vs. O(n²·d)) using the FAVOR+ random feature map:

$$\phi^+(x)_j = \frac{1}{\sqrt{m}} \exp\!\left(\omega_j^\top x - \frac{\|x\|^2}{2}\right), \quad \omega_j \sim \mathcal{N}(0, I_d)$$

This satisfies $\mathbb{E}[\phi^+(x)^\top \phi^+(y)] = \exp(x^\top y)$, so scaling $Q, K$ by $d^{-1/4}$ gives an unbiased approximation of $\exp(QK^\top / \sqrt{d})$.

The linear attention computation then becomes:

$$\text{Attn}(Q,K,V) \approx \frac{\phi(Q)\,(\phi(K)^\top V)}{\phi(Q)\,(\phi(K)^\top \mathbf{1})}$$


In [9]:
import functools
import torch
import torch.fx as fx
from performer_pytorch import FastAttention
from chop.tools import get_logger

logger = get_logger("mase_logger")

# ---------------------------------------------------------------------------
# Cache FastAttention instances (they hold the random projection matrix)
# ---------------------------------------------------------------------------
@functools.lru_cache(maxsize=None)
def _get_fast_attention(dim_heads: int, num_features: int, causal: bool, device: str):
    fa = FastAttention(dim_heads=dim_heads, nb_features=num_features, causal=causal)
    return fa.to(device)


def performer_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attn_mask=None,
    dropout_p: float = 0.0,
    is_causal: bool = False,
) -> torch.Tensor:
    """
    FAVOR+ linear attention via performer-pytorch's FastAttention.

    Accepts the same call signature as F.scaled_dot_product_attention:
        query, key, value : (B, H, N, d)
    """
    orig_dtype = query.dtype
    # FastAttention requires float32
    if orig_dtype in (torch.float16, torch.bfloat16):
        query = query.float()
        key   = key.float()
        value = value.float()

    dim_heads  = query.shape[-1]
    device_str = str(query.device)

    fa = _get_fast_attention(dim_heads, 256, is_causal, device_str)
    out = fa(query, key, value)
    return out.to(orig_dtype)


# ---------------------------------------------------------------------------
# Transform pass
# ---------------------------------------------------------------------------

def replace_sdpa_with_performer_transform_pass(mg, pass_args={}):
    """Replace scaled_dot_product_attention nodes with FAVOR+ Performer attention.

    pass_args:
        num_features (int): number of random Fourier features (default: 256).
        is_causal    (bool): use causal prefix-sum variant (default: False).
    """
    num_features = pass_args.get("num_features", 256)

    for node in list(mg.fx_graph.nodes):
        if node.op == "call_function" and "scaled_dot_product_attention" in node.name:
            logger.info(
                f"Replacing {node.target} with FAVOR+ Performer attention "
                f"(num_features={num_features}, is_causal=False)"
            )
            with mg.fx_graph.inserting_before(node):
                replacement_node = mg.fx_graph.call_function(
                    performer_attention, args=node.args, kwargs=node.kwargs
                )
            node.replace_all_uses_with(replacement_node)
            mg.fx_graph.erase_node(node)
        else:
            logger.debug(f"Skipping node: {node.target}")

    mg.fx_graph.lint()
    mg.model.recompile()
    return mg, {}


In [10]:
from chop.tools import get_tokenized_dataset
from chop.tools import get_trainer
from chop.passes.graph.utils import deepcopy_mase_graph
import copy


dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
def time_bert_model(fn, n=1000, device="cpu", dummy_input=dummy_input):
    model_device = next(fn.parameters()).device
    print(model_device)
    times = []
    labels = torch.zeros(dummy_input["input_ids"].shape[0], dtype=torch.long).to(device)
    for _ in range(n):
        if device == "cpu":
            _, t = timed_cpu(lambda: fn(dummy_input["input_ids"].cpu(), dummy_input["attention_mask"].cpu(), labels.cpu()))
        else:
            _, t = timed_gpu(lambda: fn(dummy_input["input_ids"].to(device), dummy_input["attention_mask"].to(device), labels.to(device)))
        times.append(t)
    avg_time = sum(times) / len(times)
    return avg_time

model_copy = copy.deepcopy(mg.model)


trainer = get_trainer(
    model=model_copy,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

trainer.train()

print(time_bert_model(model_copy.to(device), device=device))


# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

mg, _ = replace_sdpa_with_performer_transform_pass(mg)
mg.draw("bert-base-uncased-no_sdpa.svg")
mg.model.recompile()


INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.
/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.
/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.
/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.614300
1000,0.431200
1500,0.399600
2000,0.368000
2500,0.350700
3000,0.359700


...

INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.
/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.614300
1000,0.431200
1500,0.399600
2000,0.368000
2500,0.350700
3000,0.359700


cuda:0
0.0012356463040113443


Evaluation accuracy: 0.86152


PythonCode(src='\n\n\ndef forward(self, input_ids : torch.Tensor, attention_mask : torch.Tensor, labels : torch.Tensor):\n    size = input_ids.size()\n    getitem = size[0]\n    getitem_1 = size[1];  size = None\n    bert_embeddings_token_type_ids = self.bert.embeddings.token_type_ids\n    getitem_2 = bert_embeddings_token_type_ids[(slice(None, None, None), slice(None, getitem_1, None))];  bert_embeddings_token_type_ids = None\n    expand = getitem_2.expand(getitem, getitem_1);  getitem_2 = getitem = None\n    size_1 = input_ids.size()\n    getitem_3 = size_1[1];  size_1 = None\n    bert_embeddings_position_ids = self.bert.embeddings.position_ids\n    add = getitem_3 + 0;  getitem_3 = None\n    getitem_4 = bert_embeddings_position_ids[(slice(None, None, None), slice(0, add, None))];  bert_embeddings_position_ids = add = None\n    bert_embeddings_word_embeddings = self.bert.embeddings.word_embeddings(input_ids);  input_ids = None\n    bert_embeddings_token_type_embeddings = self.bert.em

In [11]:
from chop.tools import get_trainer

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

trainer.train()

print(time_bert_model(mg.model.to(device), device=device))


# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation accuracy: 0.50996


Step,Training Loss


/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation accuracy: 0.50996


Step,Training Loss
500,0.664600
1000,0.559700
1500,0.497900
2000,0.445800
2500,0.430200
3000,0.417700


cuda:0
0.0024541384263038644
Evaluation accuracy: 0.8222


In [13]:
print(time_bert_model(mg.model.to(device), device=device))
print(time_bert_model(model_copy.to(device), device=device))



cuda:0
0.0023499297890663116
cuda:0
0.0011889126082658768
